# Method 5 — Material-social association

**Purpose.** This pre-specified secondary analysis asks whether countries with more favourable income or employment change also had more favourable change in perceived social support or negative affect. It estimates exploratory country-level rank associations; it is neither a predictive model nor a causal design.

**Reporting boundary.** Report the four pre-specified estimates only. Negative-affect associations remain supported after Holm adjustment; social-support associations are positive but statistically less certain.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve paths from either the repository root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.oecd_audit import INDICATOR_SPECS, load_clean

AUSTRALIA = 'AUS'
BASE_SEED = 20260721
BOOTSTRAP_REPLICATES = 10_000
PERMUTATIONS = 19_999
MIN_COUNTRIES = 25
TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'

@dataclass(frozen=True)
class PairSpec:
    pair_id: str
    material_code: str
    social_code: str
    material_label: str
    social_label: str

# Freeze the four pre-specified material–social pairs before estimation.
PAIR_SPECS = [
    PairSpec('income_social_support', '1_1', '7_1_DEP', 'Income per person', 'Lack of social support'),
    PairSpec('income_negative_affect', '1_1', '11_2', 'Income per person', 'Negative affect'),
    PairSpec('employment_social_support', '2_1', '7_1_DEP', 'Employment rate', 'Lack of social support'),
    PairSpec('employment_negative_affect', '2_1', '11_2', 'Employment rate', 'Negative affect'),
]
ENDPOINTS = {
    '1_1': (2010, 2024, '2010-2024'),
    '2_1': (2010, 2024, '2010-2024'),
    '7_1_DEP': (2010, 2024, '2008-10 to 2023-25 pooled windows'),
    '11_2': (2010, 2024, '2008-10 to 2023-25 pooled windows'),
}
EXPECTED_PAIR_COUNTS = [31, 31, 43, 43]

# Load the audited tidy input rather than creating a separate cleaned dataset.
data = load_clean()
assert not data.duplicated(['country_code', 'indicator_code', 'year']).any()
assert set(code for spec in PAIR_SPECS for code in (spec.material_code, spec.social_code)) == set(ENDPOINTS)


## Construct exact, direction-consistent country changes

Only countries with both displayed endpoints enter an outcome vector. Social observations retain their three-year pooled-window interpretation; repeated rows are collapsed by `independent_period` before pivoting. Positive oriented change means improvement for all four outcomes.


In [ ]:
# Build one comparable native and oriented endpoint change for every eligible country.
def exact_endpoint_changes(frame: pd.DataFrame, code: str) -> pd.DataFrame:
    """Return one exact-endpoint native and oriented change per country."""
    start, end, period_label = ENDPOINTS[code]
    subset = frame.loc[
        frame.indicator_code.eq(code) & frame.year.isin([start, end])
    ].copy().sort_values(['country_code', 'year'])
    # Count each pooled social window once rather than as repeated annual observations.
    subset = subset.drop_duplicates(['country_code', 'independent_period'], keep='last')
    wide = (subset.pivot(index='country_code', columns='year', values='value')
            .reindex(columns=[start, end]).dropna().reset_index())
    wide = wide.rename(columns={start: 'start_value', end: 'end_value'})
    wide['native_change'] = wide.end_value - wide.start_value
    direction = INDICATOR_SPECS[code].direction
    wide['oriented_change'] = wide.native_change * (1 if direction == 'higher' else -1)
    wide['indicator_code'] = code
    wide['endpoint_period'] = period_label
    wide['better_direction'] = direction
    return wide

# Reuse these validated change vectors in every pairwise association.
changes_by_code = {code: exact_endpoint_changes(data, code) for code in ENDPOINTS}
for code, changes in changes_by_code.items():
    assert changes.country_code.is_unique
    assert changes.oriented_change.notna().all()
    sign = 1 if INDICATOR_SPECS[code].direction == 'higher' else -1
    assert (changes.oriented_change == changes.native_change * sign).all()

# Reconcile Australia with the frozen Method 2 table before any association work.
primary_path = TABLE_DIR / 'material_social_primary_results.csv'
if not primary_path.exists():
    raise FileNotFoundError('Run notebooks/02_analysis.ipynb first to generate the frozen Method 2 table.')
primary = pd.read_csv(primary_path).set_index('indicator_code')
for code, changes in changes_by_code.items():
    australia = changes.loc[changes.country_code.eq(AUSTRALIA)].iloc[0]
    assert np.isclose(australia.start_value, primary.loc[code, 'australia_start_value'])
    assert np.isclose(australia.end_value, primary.loc[code, 'australia_end_value'])
    assert np.isclose(australia.native_change, primary.loc[code, 'australia_native_change'])


## Pre-specified rank association, uncertainty and bounded sensitivities

Spearman's rho is the Pearson correlation of average ranks. Bootstrap resampling is paired at the country level. The permutation diagnostic permutes social ranks, and Holm adjustment is applied across the four pre-specified tests. Intervals and direction are primary; p-values are secondary exploratory diagnostics.


In [ ]:
# Rank both variables so the association does not assume a linear relationship.
def spearman_rho(x: np.ndarray, y: np.ndarray) -> float:
    """Spearman rho using average ranks, with an explicit tie policy."""
    x_rank = pd.Series(x).rank(method='average').to_numpy()
    y_rank = pd.Series(y).rank(method='average').to_numpy()
    if np.std(x_rank) == 0 or np.std(y_rank) == 0:
        return np.nan
    return float(np.corrcoef(x_rank, y_rank)[0, 1])

# Resample paired countries to quantify uncertainty in the country-level rank association.
def bootstrap_interval(x: np.ndarray, y: np.ndarray, rng: np.random.Generator) -> tuple[float, float, int]:
    estimates = np.empty(BOOTSTRAP_REPLICATES)
    estimates.fill(np.nan)
    for replicate in range(BOOTSTRAP_REPLICATES):
        indices = rng.integers(0, len(x), size=len(x))
        estimates[replicate] = spearman_rho(x[indices], y[indices])
    valid = estimates[np.isfinite(estimates)]
    if len(valid) < 0.99 * BOOTSTRAP_REPLICATES:
        raise RuntimeError('Fewer than 99% of country-bootstrap replicates were valid.')
    lower, upper = np.quantile(valid, [0.025, 0.975])
    return float(lower), float(upper), int(len(valid))

# Permute social ranks to provide a secondary two-sided exploratory diagnostic.
def permutation_pvalue(x: np.ndarray, y: np.ndarray, observed: float, rng: np.random.Generator) -> float:
    x_rank = pd.Series(x).rank(method='average').to_numpy()
    y_rank = pd.Series(y).rank(method='average').to_numpy()
    extreme = 0
    for _ in range(PERMUTATIONS):
        permuted = float(np.corrcoef(x_rank, rng.permutation(y_rank))[0, 1])
        extreme += abs(permuted) >= abs(observed) - 1e-12
    return float((extreme + 1) / (PERMUTATIONS + 1))

# Control the familywise error rate across the four pre-specified permutation tests.
def holm_adjust(pvalues: list[float] | pd.Series) -> np.ndarray:
    """Holm step-down adjustment in the original pair order."""
    values = np.asarray(pvalues, dtype=float)
    order = np.argsort(values)
    adjusted_sorted = np.maximum.accumulate((len(values) - np.arange(len(values))) * values[order])
    adjusted = np.empty_like(values)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted

# Exclude Australia from estimation while retaining it for the diagnostic plot.
def pairwise_changes(material_code: str, social_code: str) -> pd.DataFrame:
    material = changes_by_code[material_code][['country_code', 'oriented_change']].rename(
        columns={'oriented_change': 'material_oriented_change'}
    )
    social = changes_by_code[social_code][['country_code', 'oriented_change']].rename(
        columns={'oriented_change': 'social_oriented_change'}
    )
    return material.merge(social, on='country_code', validate='one_to_one').query('country_code != @AUSTRALIA')

all_comparator_changes = None
for code, changes in changes_by_code.items():
    column = changes[['country_code', 'oriented_change']].rename(columns={'oriented_change': code})
    column = column.loc[column.country_code.ne(AUSTRALIA)]
    all_comparator_changes = column if all_comparator_changes is None else all_comparator_changes.merge(column, on='country_code', validate='one_to_one')
assert len(all_comparator_changes) == 31


In [ ]:
# Give each pair reproducible but independent bootstrap and permutation random streams.
seed_sequences = np.random.SeedSequence(BASE_SEED).spawn(2 * len(PAIR_SPECS))
rows, plot_data = [], []
for pair_index, spec in enumerate(PAIR_SPECS):
    paired = pairwise_changes(spec.material_code, spec.social_code).copy()
    assert AUSTRALIA not in paired.country_code.values
    if len(paired) < MIN_COUNTRIES:
        raise ValueError(f'{spec.pair_id} has only {len(paired)} comparators; minimum is {MIN_COUNTRIES}.')
    x = paired.material_oriented_change.to_numpy()
    y = paired.social_oriented_change.to_numpy()
    rho = spearman_rho(x, y)
    ci_lower, ci_upper, valid_bootstrap = bootstrap_interval(x, y, np.random.default_rng(seed_sequences[2 * pair_index]))
    raw_p = permutation_pvalue(x, y, rho, np.random.default_rng(seed_sequences[2 * pair_index + 1]))
    # Check whether any single comparator reverses the association direction.
    leave_one_out = np.array([spearman_rho(np.delete(x, i), np.delete(y, i)) for i in range(len(x))])
    fixed = all_comparator_changes[['country_code', spec.material_code, spec.social_code]].dropna()
    fixed_rho = spearman_rho(fixed[spec.material_code].to_numpy(), fixed[spec.social_code].to_numpy())
    australia_material = changes_by_code[spec.material_code].loc[
        changes_by_code[spec.material_code].country_code.eq(AUSTRALIA), 'oriented_change'
    ].iloc[0]
    australia_social = changes_by_code[spec.social_code].loc[
        changes_by_code[spec.social_code].country_code.eq(AUSTRALIA), 'oriented_change'
    ].iloc[0]
    pattern = ('material improved; social outcome deteriorated'
               if australia_material > 0 and australia_social < 0
               else 'other direction pattern')
    rows.append({
        'pair_id': spec.pair_id,
        'material_indicator_code': spec.material_code,
        'social_indicator_code': spec.social_code,
        'material_outcome': spec.material_label,
        'social_outcome': spec.social_label,
        'material_endpoint_period': ENDPOINTS[spec.material_code][2],
        'social_endpoint_period': ENDPOINTS[spec.social_code][2],
        'comparator_country_count': len(paired),
        'spearman_rho': rho,
        'bootstrap_ci_lower': ci_lower,
        'bootstrap_ci_upper': ci_upper,
        'bootstrap_valid_replicates': valid_bootstrap,
        'bootstrap_valid_rate': valid_bootstrap / BOOTSTRAP_REPLICATES,
        'permutation_p_raw': raw_p,
        'fixed_common_panel_count': len(fixed),
        'fixed_common_panel_rho': fixed_rho,
        'leave_one_out_rho_min': float(np.nanmin(leave_one_out)),
        'leave_one_out_rho_max': float(np.nanmax(leave_one_out)),
        'leave_one_out_sign_stable': bool(np.all(np.sign(leave_one_out) == np.sign(rho))),
        'australia_material_oriented_change': australia_material,
        'australia_social_oriented_change': australia_social,
        'australia_direction_pattern': pattern,
    })
    paired['pair_id'] = spec.pair_id
    paired['material_label'] = spec.material_label
    paired['social_label'] = spec.social_label
    paired['rho'] = rho
    paired['ci_lower'] = ci_lower
    paired['ci_upper'] = ci_upper
    paired['australia_material'] = australia_material
    paired['australia_social'] = australia_social
    plot_data.append(paired)

# Apply the multiple-test adjustment only after all four raw p-values are available.
results = pd.DataFrame(rows)
results['permutation_p_holm'] = holm_adjust(results.permutation_p_raw)
results = results[[
    'pair_id', 'material_indicator_code', 'social_indicator_code', 'material_outcome', 'social_outcome',
    'material_endpoint_period', 'social_endpoint_period', 'comparator_country_count', 'spearman_rho',
    'bootstrap_ci_lower', 'bootstrap_ci_upper', 'bootstrap_valid_replicates', 'bootstrap_valid_rate',
    'permutation_p_raw', 'permutation_p_holm', 'fixed_common_panel_count', 'fixed_common_panel_rho',
    'leave_one_out_rho_min', 'leave_one_out_rho_max', 'leave_one_out_sign_stable',
    'australia_material_oriented_change', 'australia_social_oriented_change', 'australia_direction_pattern',
]]

assert results.pair_id.tolist() == [spec.pair_id for spec in PAIR_SPECS]
assert results.comparator_country_count.tolist() == EXPECTED_PAIR_COUNTS
assert results.comparator_country_count.ge(MIN_COUNTRIES).all()
assert results.fixed_common_panel_count.tolist() == [31, 31, 31, 31]
assert results.spearman_rho.between(-1, 1).all()
assert results.bootstrap_ci_lower.between(-1, 1).all()
assert results.bootstrap_ci_upper.between(-1, 1).all()
assert (results.bootstrap_ci_lower <= results.bootstrap_ci_upper).all()
assert results[['permutation_p_raw', 'permutation_p_holm']].apply(lambda column: column.between(0, 1).all()).all()
assert (results.permutation_p_holm >= results.permutation_p_raw).all()
assert results.bootstrap_valid_rate.ge(0.99).all()
assert results.australia_direction_pattern.eq('material improved; social outcome deteriorated').all()

# Save the frozen result table and verify that reading it back preserves every value.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
result_path = TABLE_DIR / 'material_social_spearman_results.csv'
results.to_csv(result_path, index=False)
reloaded = pd.read_csv(result_path)
pd.testing.assert_frame_equal(results, reloaded, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)
display(results.round({'spearman_rho': 3, 'bootstrap_ci_lower': 3, 'bootstrap_ci_upper': 3, 'permutation_p_holm': 4}))
print(f'Wrote and round-trip validated {result_path.relative_to(PROJECT_ROOT)}')


## Diagnostic figure

The figure shows country changes rather than fitted regression lines. Australia is displayed separately and does not contribute to estimated rho, intervals, or permutation diagnostics. The zero lines identify favourable and adverse oriented-change quadrants.


In [ ]:
# Plot estimates without a regression line because Spearman's rho is rank based.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plot_frame = pd.concat(plot_data, ignore_index=True)
material_axis_labels = {
    '1_1': 'Favourable income change\n(PPP-converted USD per person)',
    '2_1': 'Favourable employment change\n(percentage points)',
}
social_axis_labels = {
    '7_1_DEP': 'Favourable social-support change\n(percentage points)',
    '11_2': 'Favourable negative-affect change\n(percentage points)',
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for ax, spec, row in zip(axes.flat, PAIR_SPECS, results.itertuples(index=False)):
    panel = plot_frame.loc[plot_frame.pair_id.eq(spec.pair_id)]
    ax.scatter(panel.material_oriented_change, panel.social_oriented_change, color='#7A7A7A', alpha=0.7, s=34, label='Eligible comparators')
    # Highlight Australia without including it in the estimated association.
    ax.scatter(row.australia_material_oriented_change, row.australia_social_oriented_change,
               marker='D', color='#D55E00', edgecolor='black', linewidth=0.5, s=72, label='Australia', zorder=3)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.axvline(0, color='#555555', linewidth=0.8)
    ax.set_title(f'{spec.material_label} and {spec.social_label}', loc='left', fontweight='bold')
    ax.set_xlabel(material_axis_labels[spec.material_code])
    ax.set_ylabel(social_axis_labels[spec.social_code])
    ax.text(0.02, 0.98, f'Spearman ρ = {row.spearman_rho:.2f}\n95% CI [{row.bootstrap_ci_lower:.2f}, {row.bootstrap_ci_upper:.2f}]\nn = {row.comparator_country_count}',
            transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85})
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle('Exploratory cross-country associations between material and social change\nPositive values indicate improvement', fontweight='bold')
# Export the appendix-ready diagnostic figure at report resolution.
figure_path = FIGURE_DIR / 'material_social_spearman_associations.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
assert figure_path.exists() and figure_path.stat().st_size > 0
plt.show()
print(f'Wrote {figure_path.relative_to(PROJECT_ROOT)} at 300 dpi')


## Interpretation boundary

Read these estimates as exploratory ecological associations in country ordering, not as effect sizes or causal pathways. A positive rho means countries with more favourable material change tended also to have more favourable social change. It does not imply that material improvement caused social improvement, explain individual Australians, or establish the size of any social benefit per material unit. Pair-specific country composition, endpoint measurement error and pooled social windows remain material limitations.
